# ARISTA salamander brain regeneration

This notebook shows the complete `arista` analysis: data preparation,
model training, downstream analysis, and the commands used for the paper
figures. Edit the paths in **Setup** before starting a run.

## Setup

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from CytoBridge.workflow import (
    WorkflowOptions,
    build_workflow_plan,
    load_workflow_config,
    render_workflow_plan,
    run_workflow,
)
DATASET_CONFIG = 'arista'
RAW_H5AD = Path("data/arista_raw.h5ad")
OUTPUT_DIR = Path("tutorial_outputs/arista")
ALIGNED_H5AD = OUTPUT_DIR / "preprocess" / 'arista_aligned.h5ad'
MODEL_DIR = OUTPUT_DIR / "training"


RUN_PREPARATION = False
RUN_PREPROCESS_AND_TRAIN = False
RUN_DOWNSTREAM = False

In [2]:
config, config_source = load_workflow_config(DATASET_CONFIG)
dataset = config["dataset"]
scientific = config["scientific"]
downstream = config["downstream"]

pd.DataFrame(
    {
        "setting": [
            "dataset",
            "configuration",
            "raw time column",
            "cell annotation",
            "observed training times",
            "classifier neighbors",
        ],
        "value": [
            dataset["display_name"],
            config_source,
            config["preprocess"]["time_key"],
            dataset["annotation_key"],
            ", ".join(map(str, downstream["observed"])),
            scientific["classifier_k"],
        ],
    }
)

,setting,value
0,dataset,ARISTA salamander brain regeneration
1,configuration,example configuration: arista
2,raw time column,Batch
3,cell annotation,Annotation
4,observed training times,"0.0, 1.0, 2.0, 3.0, 4.0"
5,classifier neighbors,10


## Data preparation

The dataset configuration records the count layer, time mapping, spatial
coordinates, and alignment settings. The command below reads the raw H5AD and
writes the aligned H5AD and edge model used for training.

### 1. preprocess
```text
cytobridge workflow --config arista --step preprocess --input-h5ad <raw.h5ad> --output-dir <run>
```

Start with: `raw H5AD and the dataset configuration`

Writes: `<run>/preprocess/arista_aligned.h5ad; <run>/preprocess/edge_classifier/arista_edge_model.pt; preprocessing records`

Next: `training`

In [3]:
preparation_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess",),
)
preparation_plan = build_workflow_plan(
    config,
    source=config_source,
    options=preparation_options,
)
print(render_workflow_plan(preparation_plan))

CytoBridge workflow plan
dataset: ARISTA salamander brain regeneration (arista)
config: example configuration: arista
model settings: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=10
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/arista/preprocess/arista_aligned.h5ad
    edge predictor: not requested during preprocessing
  train: skipped; add --train to run (GPU required for training)
  downstream: skipped (GPU recommended)


In [4]:
if RUN_PREPARATION:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before preprocessing: {RAW_H5AD}")
    preparation_result = run_workflow(config, options=preparation_options)
    preparation_result
else:
    print("Data preparation is off. Set RUN_PREPARATION = True to run it.")

Data preparation is off. Set RUN_PREPARATION = True to run it.


## Training

The full run starts from the raw H5AD, writes the aligned data, fits the
interaction edge model when needed, and trains CytoBridge. Training requires a
CUDA-capable environment.

### 1. preprocess and train
```text
cytobridge workflow --config arista --step preprocess --step train --train --input-h5ad <raw.h5ad> --output-dir <run> --device cuda
```

Start with: `raw H5AD, dataset configuration, and LR database`

Writes: `<run>/training/<stage>/best_model.pth or score_model.pth; <run>/training/adata.h5ad; training_history.csv; training_run_summary.json`

Next: `downstream`

In [5]:
training_options = WorkflowOptions(
    input_h5ad=RAW_H5AD,
    output_dir=OUTPUT_DIR,
    steps=("preprocess", "train"),
    train=True,
)
training_plan = build_workflow_plan(
    config,
    source=config_source,
    options=training_options,
)
print(render_workflow_plan(training_plan))

CytoBridge workflow plan
dataset: ARISTA salamander brain regeneration (arista)
config: example configuration: arista
model settings: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=10
steps:
  preprocess: ready (GPU recommended for spatial alignment)
    output: tutorial_outputs/arista/preprocess/arista_aligned.h5ad
    edge predictor: will be trained automatically
      graph database: package: CytoBridge/workflow_databases/CellChatDB.ligrec.human.csv
      database source: included CellChatDB resource
      interaction cutoff: 0.03154105148551745
      decision threshold source: validation-selected during de novo training
      output: tutorial_outputs/arista/preprocess/edge_classifier/arista_edge_model.pt
  train: ready (GPU required for training)
    training config: arista_spatial_full.yaml
    interaction cutoff: 0.03154105148551745
    edge predictor threshold source: validation-selected during preprocessing
    edge predictor: tutorial_outputs/arista/preprocess/ed

In [6]:
if RUN_PREPROCESS_AND_TRAIN:
    if not RAW_H5AD.is_file():
        raise FileNotFoundError(f"Update RAW_H5AD before training: {RAW_H5AD}")
    training_result = run_workflow(config, options=training_options)
    training_result
else:
    print("Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.")

Training is off. Set RUN_PREPROCESS_AND_TRAIN = True to start it.


## Downstream analysis

Downstream analysis reads the aligned H5AD and fitted model from the training
directory. It writes generated states, velocity, growth, composition,
communication, ligand–receptor tables, and standard figures.

### 1. downstream
```text
cytobridge workflow --config arista --step downstream --aligned-h5ad <run>/preprocess/arista_aligned.h5ad --model-dir <run>/training --output-dir <run>
```

Start with: `aligned H5AD; <run>/training; dataset-matched LR database`

Writes: `<run>/downstream/summary.json; slice_data/*.h5ad; velocity/velocity_components.npz; growth/growth_by_cell.csv; composition/celltype_composition.csv; communication and ligand_receptor tables; standard figures`

Next: `paper-specific continuation shown in the paper-figure notebook`

In [7]:
downstream_options = WorkflowOptions(
    aligned_h5ad=ALIGNED_H5AD,
    model_dir=MODEL_DIR,
    output_dir=OUTPUT_DIR,
    steps=("downstream",),
)
downstream_plan = build_workflow_plan(
    config,
    source=config_source,
    options=downstream_options,
)
print(render_workflow_plan(downstream_plan))

CytoBridge workflow plan
dataset: ARISTA salamander brain regeneration (arista)
config: example configuration: arista
model settings: alpha_spatial=10, alpha_express=0.015, seed=42, classifier_k=10
steps:
  preprocess: skipped (GPU for spatial alignment)
  train: skipped; add --train to run (GPU required for training)
  downstream: ready (GPU recommended for SDE simulation and classifier fitting)
    model format: current
    output: tutorial_outputs/arista/downstream
    generated states: observed times=[0.0, 1.0, 2.0, 3.0, 4.0], additional times=[0.5, 1.5, 2.5, 3.5]
      simulation settings: dt=0.01, sigma=0.03, daughter noise=0, growth alpha=1
    interpolation and classification: enabled
    time-slice velocity: enabled
    growth: enabled when present in the model
    cell-type composition: enabled
    sparse communication: enabled
    standard figures: enabled
      note: snapshots, mosaic, growth, composition, and velocity; 3D communication only when the model has an interactio

In [8]:
if RUN_DOWNSTREAM:
    missing = [path for path in (ALIGNED_H5AD, MODEL_DIR) if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing aligned data or model directory: {missing}")
    downstream_result = run_workflow(config, options=downstream_options)
    downstream_result
else:
    print("Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.")

Downstream analysis is off. Set RUN_DOWNSTREAM = True to run it.


## Paper figures

Continue with these commands to calculate the values used in the paper. Each
step states which downstream files it reads and which paper notebook uses its
output.

- [Main Figure 5](../paper_figures/main_figure_5.ipynb)
- [Supplementary Figures S19–S24](../paper_figures/arista_figures.ipynb)
- [Supplementary Figure S42](../paper_figures/arista_local_domains.ipynb)

### 1. calculate ARISTA outputs (Main Figure 5; S19-S24; S42)
```text
cytobridge workflow --config arista --step downstream --aligned-h5ad <run>/preprocess/arista_aligned.h5ad --model-dir <run>/training --output-dir <run>
```

Start with: `aligned ARISTA H5AD and retrained six-stage model used for the paper`

Writes: `slice H5ADs; growth; composition; velocity; sparse attention; gene and strict LR tables`

Next: `run the ARISTA panel builders`

### 2. draw spatial, growth, composition, and gene-program panels (Main Figure 5; S19-S22)
Source files: `release_artifacts/arista_package_native_spatialqc_z50_retrain_20260824_r1`

Start with: `downstream outputs and the file record for each panel`

Writes: `vector panels and assembled Main Figure 5/S19-S22 pages`

Next: `use the Main Figure 5 and ARISTA paper notebooks`

The repository release contains the panel builders. The Main Figure 5 notebook also exports a viewable copy of the page. The directory is listed for reference and is not a command.

### 3. recalculate LR clusters (S23-S24)
```text
python scripts/execute_paper_notebooks.py --notebook arista_figures --output-dir <notebook-run>
```

Start with: `all 531 LR profiles`

Writes: `LR cluster and representative-pair tables`

Next: `use the ARISTA LR paper notebook`

### 4. recalculate local-domain summaries (S42)
```text
python scripts/execute_paper_notebooks.py --notebook arista_local_domains --output-dir <notebook-run>
```

Start with: `ROI, domain, edge, and null-analysis tables`

Writes: `local-domain panel tables and vector PDF/PNG`

Next: `use the ARISTA local-domain paper notebook`

## Saved files

- Aligned data: `tutorial_outputs/arista/preprocess/arista_aligned.h5ad`
- Training directory: `tutorial_outputs/arista/training`
- Downstream directory: `tutorial_outputs/arista/downstream`